# Završna provjera za modul: Strojno učenje

- **Autor: Davorin Špičko**
- **Predavač: doc. dr. sc. Kristian Đokić**

# Zadatak 1

Proučite dataset [Breast Cancer Wisconsin (Diagnostic)](https://www.kaggle.com/datasets/uciml/breast-cancer-wisconsin-data) i definirajte što je target varijabla. Ako je ne pronalazite, izaberite neku varijablu za koju postoje samo dvije vrijednosti u stupcu. Iz dataseta izbacite sve značajke koje nisu numeričke, a ako negdje nedostaje vrijednost umetnite medijan. 20% dataseta ostavite za testiranje, a pri dijeljenju dataset-a u funkciji train_test_split koristite random_state=42 i stratify=y. Istrenirajte k-NN sa 5 susjeda i ispišite koja je točnost modela. Istrenirajte i model naive bayes i ispišite točnost.

In [1]:
import pandas as pd
import numpy as np
from pandas.core.sample import sample
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LinearRegression, LogisticRegression, Lasso, Ridge
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.naive_bayes import GaussianNB
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.metrics import mean_squared_error, accuracy_score, precision_score, recall_score, f1_score
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.svm import SVC, NuSVC
from sklearn.tree import DecisionTreeClassifier
import eli5
from eli5.sklearn import PermutationImportance

In [2]:
df_cancer = pd.read_csv("breast-cancer-wisconsin-data.csv", low_memory=False)
df_cancer.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


In [3]:
df_cancer.isna().sum()

id                           0
diagnosis                    0
radius_mean                  0
texture_mean                 0
perimeter_mean               0
area_mean                    0
smoothness_mean              0
compactness_mean             0
concavity_mean               0
concave points_mean          0
symmetry_mean                0
fractal_dimension_mean       0
radius_se                    0
texture_se                   0
perimeter_se                 0
area_se                      0
smoothness_se                0
compactness_se               0
concavity_se                 0
concave points_se            0
symmetry_se                  0
fractal_dimension_se         0
radius_worst                 0
texture_worst                0
perimeter_worst              0
area_worst                   0
smoothness_worst             0
compactness_worst            0
concavity_worst              0
concave points_worst         0
symmetry_worst               0
fractal_dimension_worst      0
Unnamed:

Target varijabla je "diagnosis" koja ima dvije vrijednosti: M (malignant) i B (benign). Dataset sadrži dodatan zarez na kraju retka što rezultira dodatnim stupcem bez imena koji sadrži samo NaN vrijednosti. Taj stupac ćemo izbaciti kao i stupac sa ID-em.

In [4]:
df_cancer = df_cancer.drop(columns=['id', 'Unnamed: 32'])
df_cancer['diagnosis'] = df_cancer['diagnosis'].apply(lambda x: 1 if x == 'M' else 0)

Ovaj očišćeni dataset će se koristiti u svim zadacima vezanim uz ovaj dataset.

In [5]:
y = df_cancer['diagnosis']
X = df_cancer.drop(columns=['diagnosis'])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

models = {
    'KNN': KNeighborsClassifier(n_neighbors=5),
    'Naive Bayes': GaussianNB()
}

for name, model in models.items():
    model.fit(X_train, y_train)
    accuracy = model.score(X_test, y_test)
    print(f"{name} točnost: {accuracy:.4f}")

KNN točnost: 0.9123
Naive Bayes točnost: 0.9386


# Zadatak 2

Proučite dataset [Energy Consumption Dataset](https://www.kaggle.com/datasets/govindaramsriram/energy-consumption-dataset-linear-regression) i definirajte što je target varijabla. Iz dataseta izbacite sve značajke koje nisu numeričke, a ako negdje nedostaje vrijednost umetnite medijan. 20% dataseta ostavite za testiranje, a pri dijeljenju dataset-a u funkciji train_test_split koristite random_state=42. Istrenirajte k-NN regresiju sa 3 susjeda i ispišite koja je srednja kvadratna pogreška. Isto napravite još jednom, ali koristite linearnu regresiju.

In [6]:
df_energy_train = pd.read_csv("train_energy_data.csv", low_memory=False)
df_energy_test = pd.read_csv("test_energy_data.csv", low_memory=False)
df_energy_train.head()

,Building Type,Square Footage,Number of Occupants,Appliances Used,Average Temperature,Day of Week,Energy Consumption
0,Residential,7063,76,10,29.84,Weekday,2713.95
1,Commercial,44372,66,45,16.72,Weekday,5744.99
2,Industrial,19255,37,17,14.30,Weekend,4101.24
3,Residential,13265,14,41,32.82,Weekday,3009.14
4,Commercial,13375,26,18,11.92,Weekday,3279.17


In [7]:
df_energy_train.isna().sum()

Building Type          0
Square Footage         0
Number of Occupants    0
Appliances Used        0
Average Temperature    0
Day of Week            0
Energy Consumption     0
dtype: int64

U datasetu nema vrijednosti koje nedostaju, a target varijabla je "Energy Consumption". Izbacit ćemo kategorijske značajke "Building Type" i "Day of Week".

In [8]:
y_train = df_energy_train['Energy Consumption']
X_train = df_energy_train.drop(columns=['Energy Consumption', 'Building Type', 'Day of Week'])

y_test = df_energy_test['Energy Consumption']
X_test = df_energy_test.drop(columns=['Energy Consumption', 'Building Type', 'Day of Week'])

In [9]:
models = {
    'KNN Regression': KNeighborsRegressor(n_neighbors=3),
    'Linear Regression': LinearRegression()
}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    print(f"{name} RMSE: {rmse:.4f}")


KNN Regression RMSE: 671.0132
Linear Regression RMSE: 429.8645


# Zadatak 3

Koristite dataset [Breast Cancer Wisconsin (Diagnostic)](https://www.kaggle.com/datasets/uciml/breast-cancer-wisconsin-data). Model neka bude k-NN sa 5 susjeda. Koristite peterostruku unakrsnu provjeru modela StratifiedKFold sa parametrima n_splits=5, shuffle=True, random_state=42, a točnost dobijte korištenjem cross_val_score uz parametar scoring="accuracy". Prikažite točnosti i prosječnu vrijednost.


In [10]:
y = df_cancer['diagnosis']
X = df_cancer.drop(columns=['diagnosis'])

scaler = RobustScaler()
X_scaled = scaler.fit_transform(X)

knn = KNeighborsClassifier(n_neighbors=5)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_val_score(knn, X_scaled, y, cv=cv, scoring='accuracy')

In [11]:
print("Točnosti po presjecima (foldovima):")
for i, score in enumerate(scores, start=1):
    print(f"Presjek {i}: {score:.4f}")

print(f"\nProsječna točnost: {scores.mean():.4f}")
print(f"Standardna devijacija: {scores.std():.4f}")

Točnosti po presjecima (foldovima):
Presjek 1: 0.9825
Presjek 2: 0.9386
Presjek 3: 0.9649
Presjek 4: 0.9737
Presjek 5: 0.9558

Prosječna točnost: 0.9631
Standardna devijacija: 0.0151


# Zadatak 4

Koristite dataset [Breast Cancer Wisconsin (Diagnostic)](https://www.kaggle.com/datasets/uciml/breast-cancer-wisconsin-data). 20% dataseta ostavite za testiranje, a pri dijeljenju dataset-a u funkciji train_test_split koristite random_state=42 i stratify=y. Istrenirajte k-NN sa 5 susjeda i ispišite sve pokazatelje modela uključujući matricu konfuzije.
Nakon toga  smanjite threshhold za 50%, pa ponovo izračunajte i ispišite sve pokazatelje modela uključujući matricu konfuzije.
Nakon toga  povećajte threshhold za 50% od početnog, pa ponovo izračunajte i ispišite sve pokazatelje modela uključujući matricu konfuzije

In [12]:
def evaluate_threshold(threshold, y_true, y_proba):
    y_pred = (y_proba >= threshold).astype(int)
    print(f"\n=== Rezultati za threshold = {threshold:.2f} ===")
    print(f"Točnost:  {accuracy_score(y_true, y_pred):.4f}")
    print(f"Preciznost: {precision_score(y_true, y_pred):.4f}")
    print(f"Odziv (Recall): {recall_score(y_true, y_pred):.4f}")
    print(f"F1 score: {f1_score(y_true, y_pred):.4f}")
    print("Matrica konfuzije:")
    print(confusion_matrix(y_true, y_pred))
    print("\nDetaljan izvještaj:")
    print(classification_report(y_true, y_pred, digits=4))

In [13]:
X = df_cancer.drop(columns=['diagnosis'])
y = df_cancer['diagnosis']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_scaled, y_train)

y_proba = knn.predict_proba(X_test_scaled)[:, 1]

thresholds = np.linspace(0.5, 1, 10)
for threshold in thresholds:
    evaluate_threshold(threshold, y_test, y_proba)


=== Rezultati za threshold = 0.50 ===
Točnost:  0.9561
Preciznost: 1.0000
Odziv (Recall): 0.8810
F1 score: 0.9367
Matrica konfuzije:
[[72  0]
 [ 5 37]]

Detaljan izvještaj:
              precision    recall  f1-score   support

           0     0.9351    1.0000    0.9664        72
           1     1.0000    0.8810    0.9367        42

    accuracy                         0.9561       114
   macro avg     0.9675    0.9405    0.9516       114
weighted avg     0.9590    0.9561    0.9555       114


=== Rezultati za threshold = 0.56 ===
Točnost:  0.9561
Preciznost: 1.0000
Odziv (Recall): 0.8810
F1 score: 0.9367
Matrica konfuzije:
[[72  0]
 [ 5 37]]

Detaljan izvještaj:
              precision    recall  f1-score   support

           0     0.9351    1.0000    0.9664        72
           1     1.0000    0.8810    0.9367        42

    accuracy                         0.9561       114
   macro avg     0.9675    0.9405    0.9516       114
weighted avg     0.9590    0.9561    0.9555       114

# Zadatak 5

Koristite dataset [Breast Cancer Wisconsin (Diagnostic)](https://www.kaggle.com/datasets/uciml/breast-cancer-wisconsin-data). 20% dataseta ostavite za testiranje, a pri dijeljenju dataset-a u funkciji train_test_split koristite random_state=42 i stratify=y. Istrenirajte pet klasifikatora (diskriminantna analiza, stablo odlučivanja, SVC, NuSVC, logistička regresija) sa hiperparametrima po želji i ispišite sve pokazatelje svakog modela uključujući matricu konfuzije.

In [14]:
X = df_cancer.drop(columns=['diagnosis'])
y = df_cancer['diagnosis']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

models = {
    'LDA': LinearDiscriminantAnalysis(),
    'Decision Tree': DecisionTreeClassifier(max_depth=5, random_state=42),
    'SVC': SVC(C=1.0, kernel='linear', probability=True, random_state=42),
    'NuSVC': NuSVC(nu=0.3, kernel='rbf', probability=True, random_state=42),
    'Logistic Regression': LogisticRegression(max_iter=1000, solver='lbfgs', random_state=42)
}

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    print(f"\n=== Rezultati za model: {name} ===")
    print(f"Točnost:  {accuracy_score(y_test, y_pred):.4f}")
    print(f"Preciznost: {precision_score(y_test, y_pred):.4f}")
    print(f"Odziv (Recall): {recall_score(y_test, y_pred):.4f}")
    print(f"F1 score: {f1_score(y_test, y_pred):.4f}")
    print("Matrica konfuzije:")
    print(confusion_matrix(y_test, y_pred))
    print("\nDetaljan izvještaj:")
    print(classification_report(y_test, y_pred, digits=4))


=== Rezultati za model: LDA ===
Točnost:  0.9649
Preciznost: 1.0000
Odziv (Recall): 0.9048
F1 score: 0.9500
Matrica konfuzije:
[[72  0]
 [ 4 38]]

Detaljan izvještaj:
              precision    recall  f1-score   support

           0     0.9474    1.0000    0.9730        72
           1     1.0000    0.9048    0.9500        42

    accuracy                         0.9649       114
   macro avg     0.9737    0.9524    0.9615       114
weighted avg     0.9668    0.9649    0.9645       114


=== Rezultati za model: Decision Tree ===
Točnost:  0.9211
Preciznost: 0.9459
Odziv (Recall): 0.8333
F1 score: 0.8861
Matrica konfuzije:
[[70  2]
 [ 7 35]]

Detaljan izvještaj:
              precision    recall  f1-score   support

           0     0.9091    0.9722    0.9396        72
           1     0.9459    0.8333    0.8861        42

    accuracy                         0.9211       114
   macro avg     0.9275    0.9028    0.9128       114
weighted avg     0.9227    0.9211    0.9199       114



# Zadatak 6

Koristite dataset [Energy Consumption Dataset](https://www.kaggle.com/datasets/govindaramsriram/energy-consumption-dataset-linear-regression). Ako nema niti jedne kategorijske značajke, onda jednu numerički pretvorite u kategorijsku.
Tu jednu kategorijsku značajku konvertirajte u numeričke korištenjem OneHotEncoder-a. Model trenirajte sa linearnom regresijom, lasso regresijom i ridge regresijom. Dataset pripremite sa train_test_split funkcijom i pri tom koristite random_state=21, a 20% neka vam ostane za testiranje. Izračunajte RMSE sa testnim podacima za sva tri modela. Ispišite izračunato.


In [15]:
X = df_energy_train.drop(columns=['Energy Consumption', 'Day of Week'])
y = df_energy_train['Energy Consumption']

In [16]:
encoder = OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore')
X_encoded = encoder.fit_transform(X['Building Type'].values.reshape(-1, 1))
X = np.hstack([X.drop(columns=['Building Type']), X_encoded])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=21)

In [17]:
models = {
    "Linear Regression": LinearRegression(),
    "Lasso Regression": Lasso(alpha=0.1, random_state=21),
    "Ridge Regression": Ridge(alpha=1.0, random_state=21)
}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    print(f"{name} RMSE: {rmse:.4f}")

Linear Regression RMSE: 25.1259
Lasso Regression RMSE: 25.1443
Ridge Regression RMSE: 25.2693


# Zadatak 7

Koristite dataset [Breast Cancer Wisconsin (Diagnostic)](https://www.kaggle.com/datasets/uciml/breast-cancer-wisconsin-data). Istrenirajte model logističke regresije koristeći GridSearchCV, isprobajte promjene dva hiperparametra navedena ispod i i ispišite koje su točnosti modela, ako zadate 5 foldova u cross_val_score metodi.
```python
log_reg = LogisticRegression(
    C=0.5,          # jačina regularizacije, a tipične vrijednosti su  [0.01, 1, 10]
    penalty='l2',   # tip regularizacije, a može biti 'l1', 'elasticnet', 'none')
```
Nakon toga istrenirajte model SVC koristeći GridSearchCV, isprobajte promjene dva hiperparametra navedena ispod i ispišite koje su točnosti modela, ako zadate 5 foldova u cross_val_score metodi. Evo hiperparametara u nastavku:
```python
svc = SVC(
    C=1.0,          # jačina regularizacije, a tipične vrijednosti su  [0.01, 1, 10]
    kernel='linear',# linearni kernel, a može biti 'poly', 'rbf', 'sigmoid')
```

In [18]:
X = df_cancer.drop(columns=['diagnosis'])
y = df_cancer['diagnosis']

scaler = RobustScaler()
X_scaled = scaler.fit_transform(X)

log_reg = LogisticRegression(max_iter=10000)

param_grid = {
    'C': np.logspace(0.01, 100, 100),
    'penalty': ['l2']
}

grid_search = GridSearchCV(log_reg, param_grid, cv=5, scoring='accuracy')
grid_search.fit(X_scaled, y)

results = pd.DataFrame(grid_search.cv_results_)
for mean_score, params in zip(results['mean_test_score'], results['params']):
    print(f"Točnost: {mean_score:.4f} za parametre: {params}")

print("\n=== Najbolji model ===")
print(f"Najbolji parametri: {grid_search.best_params_}")
print(f"Najbolja točnost (5-fold CV): {grid_search.best_score_:.4f}")

Točnost: 0.9807 za parametre: {'C': np.float64(1.023292992280754), 'penalty': 'l2'}
Točnost: 0.9702 za parametre: {'C': np.float64(10.471285480508996), 'penalty': 'l2'}
Točnost: 0.9667 za parametre: {'C': np.float64(107.1519305237606), 'penalty': 'l2'}
Točnost: 0.9596 za parametre: {'C': np.float64(1096.4781961431852), 'penalty': 'l2'}
Točnost: 0.9631 za parametre: {'C': np.float64(11220.18454301963), 'penalty': 'l2'}
Točnost: 0.9631 za parametre: {'C': np.float64(114815.36214968817), 'penalty': 'l2'}
Točnost: 0.9596 za parametre: {'C': np.float64(1174897.5549395303), 'penalty': 'l2'}
Točnost: 0.9631 za parametre: {'C': np.float64(12022644.34617413), 'penalty': 'l2'}
Točnost: 0.9596 za parametre: {'C': np.float64(123026877.0812381), 'penalty': 'l2'}
Točnost: 0.9596 za parametre: {'C': np.float64(1258925411.794166), 'penalty': 'l2'}
Točnost: 0.9614 za parametre: {'C': np.float64(12882495516.931322), 'penalty': 'l2'}
Točnost: 0.9596 za parametre: {'C': np.float64(131825673855.64047), 'pe

# Zadatak 8

Koristite dataset [Breast Cancer Wisconsin (Diagnostic)](https://www.kaggle.com/datasets/uciml/breast-cancer-wisconsin-data). Istrenirajte ansambl modele Random Forest i Gradient Boost. Standardizacija u ovom slučaju nije potrebna jer ako se ne zada bazni model, koristi se stablo odlučivanja koje ne zahtijeva standardizaciju. Koristite GridSearcCV, a mijenjajte samo jedan hiperparametar po želji (dvije vrijednosti) i ispišite točnosti.

In [19]:
X = df_cancer.drop(columns=['diagnosis'])
y = df_cancer['diagnosis']

models = {
    'Random Forest': RandomForestClassifier(random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42)
}

param_grid_rf = {
    'n_estimators': np.linspace(20, 200, 10).astype(int)
}
param_grid_gb = {
    'learning_rate': np.linspace(0.01, 1, 10)
}

grid_search_rf = GridSearchCV(models['Random Forest'], param_grid_rf, cv=5, scoring='accuracy')

grid_search_rf.fit(X, y)
results_rf = pd.DataFrame(grid_search_rf.cv_results_)

grid_search_gb = GridSearchCV(models['Gradient Boosting'], param_grid_gb, cv=5, scoring='accuracy')
grid_search_gb.fit(X, y)
results_gb = pd.DataFrame(grid_search_gb.cv_results_)

In [20]:
print("=== Random Forest rezultati ===")
for mean_score, params in zip(results_rf['mean_test_score'], results_rf['params']):
    print(f"Točnost: {mean_score:.4f} za parametre: {params}")
print("\n=== Najbolji Random Forest model ===")
print(f"Najbolji parametri: {grid_search_rf.best_params_}")
print(f"Najbolja točnost (5-fold CV): {grid_search_rf.best_score_:.4f}")


print("\n=== Gradient Boosting rezultati ===")
for mean_score, params in zip(results_gb['mean_test_score'], results_gb['params']):
    print(f"Točnost: {mean_score:.4f} za parametre: {params}")
print("\n=== Najbolji Gradient Boosting model ===")
print(f"Najbolji parametri: {grid_search_gb.best_params_}")
print(f"Najbolja točnost (5-fold CV): {grid_search_gb.best_score_:.4f}")

=== Random Forest rezultati ===
Točnost: 0.9561 za parametre: {'n_estimators': np.int64(20)}
Točnost: 0.9561 za parametre: {'n_estimators': np.int64(40)}
Točnost: 0.9578 za parametre: {'n_estimators': np.int64(60)}
Točnost: 0.9543 za parametre: {'n_estimators': np.int64(80)}
Točnost: 0.9561 za parametre: {'n_estimators': np.int64(100)}
Točnost: 0.9543 za parametre: {'n_estimators': np.int64(120)}
Točnost: 0.9561 za parametre: {'n_estimators': np.int64(140)}
Točnost: 0.9561 za parametre: {'n_estimators': np.int64(160)}
Točnost: 0.9543 za parametre: {'n_estimators': np.int64(180)}
Točnost: 0.9543 za parametre: {'n_estimators': np.int64(200)}

=== Najbolji Random Forest model ===
Najbolji parametri: {'n_estimators': np.int64(60)}
Najbolja točnost (5-fold CV): 0.9578

=== Gradient Boosting rezultati ===
Točnost: 0.9438 za parametre: {'learning_rate': np.float64(0.01)}
Točnost: 0.9614 za parametre: {'learning_rate': np.float64(0.12)}
Točnost: 0.9649 za parametre: {'learning_rate': np.float6

# Zadatak 9

Koristite dataset [Breast Cancer Wisconsin (Diagnostic)](https://www.kaggle.com/datasets/uciml/breast-cancer-wisconsin-data). Istrenirajte SVC modele za kernelima poly i rbf, ali za svaki izaberite po jedan hiperparametar i isprobajte po dvije vrijednosti koristeći GridSearchCV. Ispišite koje koji model je najbolji, odnosno hiperparametre najboljeg modela.

In [21]:
X = df_cancer.drop(columns=['diagnosis'])
y = df_cancer['diagnosis']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

models = {
    'SVC_poly': SVC(kernel='poly', probability=True, random_state=42),
    'SVC_rbf': SVC(kernel='rbf', probability=True, random_state=42)
}

param_grid_poly = {
    'degree': [2, 3]
}
param_grid_rbf = {
    'gamma': ['scale', 'auto']
}
grid_search_poly = GridSearchCV(models['SVC_poly'], param_grid_poly, cv=5, scoring='accuracy')
grid_search_poly.fit(X_scaled, y)
results_poly = pd.DataFrame(grid_search_poly.cv_results_)

grid_search_rbf = GridSearchCV(models['SVC_rbf'], param_grid_rbf, cv=5, scoring='accuracy')
grid_search_rbf.fit(X_scaled, y)
results_rbf = pd.DataFrame(grid_search_rbf.cv_results_)

print("=== SVC Poly rezultati ===")
for mean_score, params in zip(results_poly['mean_test_score'], results_poly['params']):
    print(f"Točnost: {mean_score:.4f} za parametre: {params}")
print("\n=== Najbolji SVC Poly model ===")
print(f"Najbolji parametri: {grid_search_poly.best_params_}")
print(f"Najbolja točnost (5-fold CV): {grid_search_poly.best_score_:.4f}")

print("\n=== SVC RBF rezultati ===")
for mean_score, params in zip(results_rbf['mean_test_score'], results_rbf['params']):
    print(f"Točnost: {mean_score:.4f} za parametre: {params}")
print("\n=== Najbolji SVC RBF model ===")
print(f"Najbolji parametri: {grid_search_rbf.best_params_}")
print(f"Najbolja točnost (5-fold CV): {grid_search_rbf.best_score_:.4f}")

if grid_search_poly.best_score_ > grid_search_rbf.best_score_:
    best_model = 'SVC Poly'
    best_params = grid_search_poly.best_params_
    best_score = grid_search_poly.best_score_
else:
    best_model = 'SVC RBF'
    best_params = grid_search_rbf.best_params_
    best_score = grid_search_rbf.best_score_

print(f"\nNajbolji model je {best_model} s parametrima {best_params} i točnošću {best_score:.4f}")

=== SVC Poly rezultati ===
Točnost: 0.8102 za parametre: {'degree': 2}
Točnost: 0.8981 za parametre: {'degree': 3}

=== Najbolji SVC Poly model ===
Najbolji parametri: {'degree': 3}
Najbolja točnost (5-fold CV): 0.8981

=== SVC RBF rezultati ===
Točnost: 0.9736 za parametre: {'gamma': 'scale'}
Točnost: 0.9736 za parametre: {'gamma': 'auto'}

=== Najbolji SVC RBF model ===
Najbolji parametri: {'gamma': 'scale'}
Najbolja točnost (5-fold CV): 0.9736

Najbolji model je SVC RBF s parametrima {'gamma': 'scale'} i točnošću 0.9736


# Zadatak 10

Koristeći biblioteku ELI5 prikažite ukupan utjecaj pojedinih značajki modela Random Forest koji kreirajte koristeći bolji hiperparametar otkriven u zadatku #8. Osim toga prikažite utjecaj pojedine značajke za prvi primjer u testnom datasetu.

In [22]:
X = df_cancer.drop(columns=['diagnosis'])
y = df_cancer['diagnosis']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

rf = RandomForestClassifier(n_estimators=60, random_state=42)
rf.fit(X_train, y_train)

perm = PermutationImportance(rf, random_state=42).fit(X_test, y_test)

In [23]:
print("Ukupan utjecaj značajki:")
display(eli5.show_weights(perm, feature_names=X.columns.tolist()))

Ukupan utjecaj značajki:


Weight,Feature
0.0246 ± 0.0131,concave points_worst
0.0158 ± 0.0205,area_worst
0.0088 ± 0.0111,radius_worst
0.0070 ± 0.0131,texture_worst
0.0053 ± 0.0179,perimeter_worst
0.0035 ± 0.0140,concave points_mean
0.0035 ± 0.0086,texture_mean
0 ± 0.0000,fractal_dimension_worst
0 ± 0.0000,texture_se
0.0000 ± 0.0111,concavity_worst


In [24]:
print("Utjecaj značajki za prvi primjer u testnom datasetu:")
display(eli5.show_prediction(rf, X_test.iloc[0], feature_names=X.columns.tolist()))

Utjecaj značajki za prvi primjer u testnom datasetu:


/home/davorin/workspaces/ai-centar-lipik/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2742: UserWarning: X has feature names, but DecisionTreeClassifier was fitted without feature names
  warnings.warn(
/home/davorin/workspaces/ai-centar-lipik/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2742: UserWarning: X has feature names, but DecisionTreeClassifier was fitted without feature names
  warnings.warn(
/home/davorin/workspaces/ai-centar-lipik/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2742: UserWarning: X has feature names, but DecisionTreeClassifier was fitted without feature names
  warnings.warn(
/home/davorin/workspaces/ai-centar-lipik/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2742: UserWarning: X has feature names, but DecisionTreeClassifier was fitted without feature names
  warnings.warn(
/home/davorin/workspaces/ai-centar-lipik/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2742: UserWa